In [0]:
# ==========================================
# STEP 1: BRONZE LAYER - Auto Loader Ingestion
# ==========================================

# 1. Define your exact S3 paths
raw_s3_path = "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/raw/"
bronze_s3_path = "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/bronze/"
checkpoint_path = "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/checkpoints/bronze_ingest/"
schema_path = "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/schema_data/bronze_schema/"

print("Starting Auto Loader ingestion stream...")

# 2. Configure Auto Loader to read the CSVs incrementally
raw_stream_df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true") 
    .option("cloudFiles.schemaLocation", schema_path) 
    .option("header", "true")
    .load(raw_s3_path)
)

# 3. Add Data Engineering Audit Columns
from pyspark.sql.functions import current_timestamp, input_file_name

enriched_raw_df = raw_stream_df \
    .withColumn("_ingest_timestamp", current_timestamp()) \
    .withColumn("_source_file", input_file_name())

# 4. Write out to the Bronze folder as a Delta Table using 'availableNow=True'
print(f"Writing to Bronze Delta table at {bronze_s3_path} ...")

query = (
    enriched_raw_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True) # This ensures it processes everything once and shuts down (Cost Savings!)
    .start(bronze_s3_path)
)

# Wait for the stream to finish processing the batch
query.awaitTermination()

print("✅ Bronze ingestion complete! Raw data is now secured in Delta format.")

# 5. Verify what we wrote
display(spark.read.format("delta").load(bronze_s3_path))